# Optimizing LLM-Powered Applications: Cost, Tokens & Performance

Run the cell below to set up (installs, configures, and pings the model).
That's the only setup you do - the rest of the notebook follows.


In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/shrijayan/coding-agent.git"
REPO_BRANCH = "main"

BASE_DIR = os.getcwd()                       # /content on Colab
REPO_DIR = os.path.join(BASE_DIR, "coding-agent")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning", REPO_URL, "...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    # Already cloned (e.g. re-running this cell in an existing Colab
    # session) - pull so a fast-moving repo's latest fixes actually land
    # here instead of silently running a stale checkout indefinitely.
    print("Repo already present at", REPO_DIR, "- pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# Make `import coding_agent` work without a full package install.
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("REPO_DIR =", REPO_DIR)


# ------------------------------------------------------------------
# Setup complete.
# ------------------------------------------------------------------

import subprocess, sys

# Install the project (and its declared dependencies) from its own manifest.
# `.[notebook]` = the repo's runtime deps + the notebook-only extras
# (pandas/matplotlib) declared under [project.optional-dependencies] in
# pyproject.toml. cwd=REPO_DIR is the "change into the repository directory
# first" step. Nothing about which packages to install lives in this notebook.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=REPO_DIR,
    check=True,
)

print("Dependencies installed from", REPO_DIR + "/pyproject.toml")


# ------------------------------------------------------------------
# Setup complete.
# ------------------------------------------------------------------

# Run this cell. It asks for provider + model in the terminal.
# API key still comes only from a Colab secret (never typed here).
import os

PROVIDERS = {
    "openrouter": {
        "env_var": "OPENROUTER_API_KEY",
        "default_model": "deepseek/deepseek-v4-flash-0731",
        "presets": {
            "low": "google/gemma-3.4b",
            "medium": "qwen/qwen3.7-flash",
            "high": "deepseek/deepseek-v4-flash-0731",
        },
        "keys_url": "https://openrouter.ai/keys",
    },
    "anthropic": {
        "env_var": "ANTHROPIC_API_KEY",
        "default_model": "claude-sonnet-5",
        "presets": {},
        "keys_url": "https://console.anthropic.com/settings/keys",
    },
}

provider = input("Provider [openrouter/anthropic] (default openrouter): ").strip().lower() or "openrouter"
assert provider in PROVIDERS, (
    f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}."
)
pcfg = PROVIDERS[provider]

_model_choice = input(
    f"Model (blank = {pcfg['default_model']}; openrouter also accepts "
    "low/medium/high): "
).strip()
if not _model_choice:
    model = pcfg["default_model"]
elif _model_choice.lower() in pcfg["presets"]:
    model = pcfg["presets"][_model_choice.lower()]
else:
    model = _model_choice

def _load_api_key(env_var):
    """Read the selected provider's key from a Colab secret, never from a paste."""
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(env_var)
        if val:
            return val.strip(), f"Colab secret '{env_var}'"
    except Exception:
        pass
    raise RuntimeError(
        f"No Colab secret '{env_var}' found. Add it in the Secrets panel "
        f"(left sidebar -> key icon -> '+ New secret'), then re-run this cell. "
        f"Get a key at {pcfg['keys_url']}."
    )

key, key_src = _load_api_key(pcfg["env_var"])
assert key, f"No API key for {provider}. Get one at {pcfg['keys_url']}."
os.environ[pcfg["env_var"]] = key

os.environ["AGENT_PROVIDER"] = provider
os.environ["AGENT_MODEL"] = model
os.environ["AGENT_SUMMARY_THRESHOLD_MESSAGES"] = "8"

with open(os.path.join(REPO_DIR, ".env.example")) as _f:
    for _line in _f:
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _key, _, _value = _line.partition("=")
        _key, _value = _key.strip(), _value.strip()
        if _value:
            os.environ.setdefault(_key, _value)

print(f"Provider : {provider}")
print(f"Model    : {model}")
print(f"API key  : loaded from {key_src} (length {len(key)})")


# ------------------------------------------------------------------
# Setup complete.
# ------------------------------------------------------------------

from coding_agent.config import Config
from coding_agent.metrics.pricing import PricingTable
from coding_agent.models_config import read_models_yaml
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS

cfg = Config.from_env()
pricing = PricingTable.load()
pricing.require(cfg.model)

print("Base model      :", f"{cfg.provider} / {cfg.model}")
print("Session cost cap :", cfg.session_cost_cap_usd, "USD")
print("Optimizations available:", ", ".join(AVAILABLE_OPTIMIZATIONS) or "none")

# Live ping through the agent's own client.
from coding_agent.llm.factory import build_llm_client
from coding_agent.llm.messages import Message, TextPart

_client = build_llm_client(cfg)
try:
    _r = _client.send(
        system="Reply with the single word: ok",
        messages=[Message(role="user", parts=[TextPart("ping")])],
        tools=[],
    )
    print("\nConnectivity OK — model replied:", repr(_r.text.strip()[:60]))
    print("Reported usage :", _r.usage)
except Exception as e:
    print("\nConnectivity FAILED:", type(e).__name__, "-", str(e)[:300])
    print("Check the API key for your provider, and that MODEL in Step 3 is a")
    print("valid slug for it (OpenRouter: low/medium/high preset or a real slug")
    print("from https://openrouter.ai/models; Anthropic: e.g. claude-sonnet-5).")


# ------------------------------------------------------------------
# Setup complete.
# ------------------------------------------------------------------

import os, shutil, time
from pathlib import Path

from coding_agent.config import Config
from coding_agent.metrics.pricing import PricingTable
from coding_agent.metrics.usage import UsageTracker
from coding_agent.agent.factory import build_agent
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS
from coding_agent.optimizations.registry import OptimizationRegistry
from coding_agent.commands.usage_command import UsageCommand
from coding_agent.models_config import load_catalog_metadata, read_models_yaml

# ANSI colors so the output is easy to read (Jupyter/Colab render these). Same
# scheme as the terminal app: the agent's answer is GREEN, your prompt WHITE,
# and every auxiliary line (per-turn summary, tool calls, /usage, /metrics,
# /cache, warnings) YELLOW; errors RED.
_GREEN, _WHITE, _YELLOW, _RED, _RESET = "\033[32m", "\033[37m", "\033[33m", "\033[31m", "\033[0m"

def _c(text, code):
    """Wrap text in an ANSI color for the notebook's rendered output."""
    return f"{code}{text}{_RESET}"

PLAYGROUND = Path(BASE_DIR) / "playground"

def reset_playground():
    """Wipe and recreate the agent's scratch directory, then move into it."""
    if PLAYGROUND.exists():
        shutil.rmtree(PLAYGROUND)
    PLAYGROUND.mkdir(parents=True, exist_ok=True)
    os.chdir(PLAYGROUND)

class WorkshopSession:
    """An agent session with a chosen set of optimizations enabled."""

    def __init__(self, optimizations=None, enforce_cost_cap=False):
        self.enabled = list(optimizations or [])
        self.config = Config.from_env()
        self.pricing = PricingTable.load()
        self.pricing.require(self.config.model)
        self.usage = UsageTracker()

        bundle = OptimizationRegistry(AVAILABLE_OPTIMIZATIONS).resolve(self.enabled)
        self.agent = build_agent(
            self.config, self.usage, bundle,
            pricing=self.pricing if enforce_cost_cap else None,
        )

        self.configured_models = None
        self.routing_tracker = None
        if "hybrid-routing" in self.enabled:
            from coding_agent.optimizations import hybrid_routing
            from coding_agent.optimizations.routing.tiers import load_tiers
            tiers = load_tiers(provider=self.config.provider)
            for t in tiers:
                self.pricing.require(t.model or self.config.model)
            self.configured_models = [t.model or self.config.model for t in tiers]
            self.routing_tracker = hybrid_routing.get_tracker()
            for w in hybrid_routing.get_warnings():
                print(_c(f"warning> {w}", _YELLOW))

        # Cache-friendly construction records into its own shared tracker,
        # captured here per-session exactly like the routing tracker above.
        self.cache_tracker = None
        if "cache-friendly-prompts" in self.enabled:
            from coding_agent.optimizations import cache_friendly
            self.cache_tracker = cache_friendly.get_tracker()

        self.loop_guard_tracker = None
        if "loop-guard" in self.enabled:
            from coding_agent.optimizations import loop_guard
            self.loop_guard_tracker = loop_guard.get_tracker()

        self.context_tracker = None
        if "context-window" in self.enabled:
            from coding_agent.optimizations import context_window
            self.context_tracker = context_window.get_tracker()

        # The three prompt-optimization techniques (Optimization 2a) each
        # record into their own shared tracker, same pattern as above.
        self.tool_filter_tracker = None
        if "tool-filtering" in self.enabled:
            from coding_agent.optimizations import tool_filtering
            self.tool_filter_tracker = tool_filtering.get_tracker()

        self.compression_tracker = None
        if "prompt-compression" in self.enabled:
            from coding_agent.optimizations import prompt_compression
            self.compression_tracker = prompt_compression.get_tracker()

        self.dedup_tracker = None
        if "deduplication" in self.enabled:
            from coding_agent.optimizations import deduplication
            self.dedup_tracker = deduplication.get_tracker()

        self._meta = load_catalog_metadata(read_models_yaml())

    def ask(self, prompt, show_tools=True):
        """Run one turn: print the tool calls and the final answer."""
        PLAYGROUND.mkdir(parents=True, exist_ok=True)
        os.chdir(PLAYGROUND)
        start = time.perf_counter()

        def on_tool(name, tool_input):
            if show_tools:
                print(_c(f"  [tool] {name}({tool_input})", _YELLOW))

        print(_c(f"you> {prompt}", _WHITE))
        answer = self.agent.run_turn(prompt, on_tool_call=on_tool)
        ms = (time.perf_counter() - start) * 1000
        print(_c(f"\nagent> {answer}", _GREEN))
        summary = (f"  \u21b3 {self.config.model} \u00b7 {self.usage.llm_calls} LLM calls "
                   f"(session) \u00b7 {self.usage.total.total_tokens:,} tokens (session) "
                   f"\u00b7 {ms:.0f}ms this turn")
        print(_c(summary, _YELLOW) + "\n")
        return answer

    def usage_report(self):
        """Print the same thing the /usage terminal command prints."""
        cmd = UsageCommand(
            tracker=self.usage, pricing=self.pricing, config=self.config,
            enabled_optimizations=self.enabled,
            configured_models=self.configured_models,
            model_metadata=self._meta,
            cost_cap_usd=self.config.session_cost_cap_usd,
        )
        print(_c(cmd.run(), _YELLOW))

    def routing_report(self):
        """Print the /metrics command output (only meaningful with routing)."""
        if not self.routing_tracker:
            print("Routing is not enabled for this session.")
            return
        from coding_agent.commands.metrics_command import RoutingMetricsCommand
        report = RoutingMetricsCommand(tracker=self.routing_tracker, pricing=self.pricing).run()
        print(_c(report, _YELLOW))

    def cache_report(self):
        """Print the /cache command output (only with cache-friendly-prompts)."""
        if not self.cache_tracker:
            print("Cache-friendly prompt construction is not enabled for this session.")
            return
        from coding_agent.commands.cache_command import PromptCacheCommand
        print(_c(PromptCacheCommand(tracker=self.cache_tracker).run(), _YELLOW))

    def loop_guard_report(self):
        """Print the /loopguard command output (only with loop-guard)."""
        if not self.loop_guard_tracker:
            print("Loop guard is not enabled for this session.")
            return
        from coding_agent.commands.loop_guard_command import LoopGuardCommand
        print(_c(LoopGuardCommand(tracker=self.loop_guard_tracker).run(), _YELLOW))

    def context_report(self):
        """Print the /context command output (only with context-window)."""
        if not self.context_tracker:
            print("Context window optimization is not enabled for this session.")
            return
        from coding_agent.commands.context_command import ContextWindowCommand
        print(_c(ContextWindowCommand(tracker=self.context_tracker).run(), _YELLOW))

    def tool_filter_report(self):
        """Print the /toolfilter command output (only with tool-filtering)."""
        if not self.tool_filter_tracker:
            print("Tool filtering is not enabled for this session.")
            return
        from coding_agent.commands.tool_filter_command import ToolFilterCommand
        print(_c(ToolFilterCommand(tracker=self.tool_filter_tracker).run(), _YELLOW))

    def compression_report(self):
        """Print the /compression command output (only with prompt-compression)."""
        if not self.compression_tracker:
            print("Prompt compression is not enabled for this session.")
            return
        from coding_agent.commands.compression_command import CompressionCommand
        print(_c(CompressionCommand(tracker=self.compression_tracker).run(), _YELLOW))

    def dedup_report(self):
        """Print the /dedup command output (only with deduplication)."""
        if not self.dedup_tracker:
            print("Deduplication is not enabled for this session.")
            return
        from coding_agent.commands.dedup_command import DedupCommand
        print(_c(DedupCommand(tracker=self.dedup_tracker).run(), _YELLOW))

    def metrics(self, label=None):
        """The measured numbers for this session, as a plain dict."""
        total = self.usage.total
        cost = sum(self.pricing.cost_for(u, m) for m, u in self.usage.by_model.items())
        return {
            "scenario": label or (", ".join(self.enabled) or "base"),
            "optimizations": ", ".join(self.enabled) or "none",
            "llm_calls": self.usage.llm_calls,
            "tool_calls": self.usage.tool_calls,
            "input_tokens": total.input_tokens,
            "output_tokens": total.output_tokens,
            "total_tokens": total.total_tokens,
            "cost_usd": round(cost, 6),
        }

print("Harness ready: WorkshopSession, reset_playground().")


# ------------------------------------------------------------------
# Setup complete.
# ------------------------------------------------------------------

def run_scenario(label, optimizations, prompts, show_tools=False, setup=None):
    """Run the SAME prompts through a fresh agent with the chosen optimizations.

    setup, if given, runs right after the playground is wiped and before the
    session starts - e.g. seeding a fixture file every scenario should see.
    """
    reset_playground()
    if setup is not None:
        setup()
    session = WorkshopSession(optimizations=optimizations)
    opt_label = ", ".join(optimizations) or "none"
    print(_c(f"=== Scenario: {label}  (optimizations: {opt_label}) ===", _YELLOW))
    for p in prompts:
        session.ask(p, show_tools=show_tools)
    m = session.metrics(label)
    print(_c(f"--- {label}: {m['total_tokens']:,} tokens \u00b7 ${m['cost_usd']:.4f} "
             f"\u00b7 {m['llm_calls']} LLM calls ---", _YELLOW) + "\n")
    return {"session": session, "metrics": m}

def _pct(base, new):
    return 0.0 if base == 0 else (base - new) / base * 100.0

def compare(baseline, *others):
    """Show baseline vs one or more optimized runs, with % saved."""
    rows = [baseline["metrics"]] + [o["metrics"] for o in others]
    base = baseline["metrics"]
    try:
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(rows)
        df["tokens_saved_%"] = df["total_tokens"].apply(
            lambda t: round(_pct(base["total_tokens"], t), 1))
        df["cost_saved_%"] = df["cost_usd"].apply(
            lambda c: round(_pct(base["cost_usd"], c), 1))
        cols = ["scenario", "llm_calls", "tool_calls", "total_tokens",
                "tokens_saved_%", "cost_usd", "cost_saved_%"]
        display(df[cols])
        return df
    except Exception:
        print(_c(f"{'scenario':<26}{'tokens':>10}{'saved%':>9}{'cost$':>11}{'saved%':>9}", _YELLOW))
        for r in rows:
            print(_c(f"{r['scenario']:<26}{r['total_tokens']:>10,}"
                     f"{_pct(base['total_tokens'], r['total_tokens']):>8.1f}%"
                     f"{r['cost_usd']:>11.4f}"
                     f"{_pct(base['cost_usd'], r['cost_usd']):>8.1f}%", _YELLOW))

print("Ready: run_scenario(...), compare(...).")


---
## Meet the base agent

No optimizations yet. Watch the `[tool]` lines - the agent really reads/writes files and runs commands. Edit the prompt and re-run.


In [ ]:
reset_playground()
base_demo = WorkshopSession()   # no optimizations
base_demo.ask("Create hello.py that prints 'Hello, workshop!', then show me its contents.")

### The `/usage` command

Prints tokens, cost, and active optimizations for the session above.


In [ ]:
base_demo.usage_report()

---
## The shared prompt set

- Every configuration runs the **same fixed conversation** (builds a small module over several turns).
- The task never changes - only the optimization does - so any difference is caused by the optimization.


In [ ]:
DEMO_PROMPTS = [
    "Create a file calculator.py with a function add(a, b) that returns their sum. Keep it minimal.",
    "Add a subtract(a, b) function to calculator.py.",
    "Add multiply(a, b) and divide(a, b) to calculator.py. divide must raise ValueError on division by zero.",
    "Create test_calculator.py with pytest tests for all four functions, including the divide-by-zero case.",
    "Read calculator.py and test_calculator.py again in full to double-check them, and confirm the tests cover every function.",
    "List the files you created and give a one-line summary of what calculator.py now contains.",
]
print(f"{len(DEMO_PROMPTS)} prompts ready.")

---
## Baseline - the un-optimized agent

Runs the shared prompts with **no** optimizations and records the numbers. Every later scenario is measured against this. (Calls the model several times - give it a moment.)


In [ ]:
baseline = run_scenario("base agent", [], DEMO_PROMPTS)
baseline["session"].usage_report()

---
## Optimization 1 - Conversation summarization

- Every turn resends the **entire** history; older messages are billed again on every call.
- Summarization compresses old messages into a short running summary; recent messages stay verbatim.

*Terminal equivalent:* `uv run coding-agent --enable conversation-summary`


In [ ]:
opt_summary = run_scenario("+ conversation-summary", ["conversation-summary"], DEMO_PROMPTS)
compare(baseline, opt_summary)

---
## Optimization 2 - Prompt optimization & caching

Two levers:

- **2a . Prompt optimization** - send *fewer* tokens (a family of five techniques).
- **2b . Cache-friendly prompts** - make the tokens that must repeat *cheaper to resend*.


### 2a - Prompt optimization: the five techniques

Five techniques, each removing a different kind of waste. All live in the repo, each behind its own flag.

| Technique | What it removes | Flag | Measured |
|---|---|---|---|
| **Context pruning** | irrelevant conversation history (stale bulky tool output) | `context-window` | Optimization 5 |
| **Conversation summarization** | older messages, replaced by a concise summary | `conversation-summary` | Optimization 1 |
| **Tool filtering** | tool definitions irrelevant to the request | `tool-filtering` | **here** |
| **Prompt compression** | long instructions, rewritten shorter | `prompt-compression` | **here** |
| **Deduplication** | identical content repeated in one request | `deduplication` | **here** |

Removals are deterministic and local - never a model call to save tokens.


#### Anatomy of a request - what gets sent every call

Every call carries: system prompt, tool definitions, conversation so far, latest turn - each tagged by volatility, which decides what's optimized (2a) and what's cacheable (2b).

| Layer | Volatility | Lever |
|:---|:---:|---|
| System prompt | `stable` | prompt compression |
| Tool definitions | `stable` | tool filtering + prompt compression |
| Conversation so far | `semi-stable` | summarization . pruning . deduplication |
| Latest message & tool results | `dynamic` | the only new tokens |

The next cell prints the real system prompt, a tool definition, and the layered build.


In [ ]:
import json

from coding_agent.system_prompt import SYSTEM_PROMPT, SYSTEM_PROMPT_COMPACT
from coding_agent.tools.registry import ToolRegistry
from coding_agent.tools.read_file import ReadFileTool
from coding_agent.tools.write_file import WriteFileTool
from coding_agent.tools.edit_file import EditFileTool
from coding_agent.tools.bash import BashTool
from coding_agent.tools.list_files import ListFilesTool
from coding_agent.llm.messages import Message, TextPart, ToolResultPart, ToolUsePart
from coding_agent.optimizations.prompt_cache.builder import PromptBuilder

# The exact tool set agent/factory.py builds (bash timeout from the same config).
_tools = ToolRegistry(tools=[
    ReadFileTool(), WriteFileTool(), EditFileTool(),
    BashTool(timeout_seconds=cfg.bash_timeout_seconds), ListFilesTool(),
]).definitions()

print(_c("=== STABLE · the system prompt (verbatim, sent with EVERY call) ===", _YELLOW))
print(SYSTEM_PROMPT)
print(_c(f"--- its hand-tightened equivalent, swapped in by --enable prompt-compression "
         f"({len(SYSTEM_PROMPT)} -> {len(SYSTEM_PROMPT_COMPACT)} chars, same rules) ---", _YELLOW))
print(SYSTEM_PROMPT_COMPACT)

print(_c(f"=== STABLE · tool definitions ({len(_tools)} tools, also sent with EVERY call) ===", _YELLOW))
print(_c(f"one of them, verbatim (tool filtering withholds irrelevant ones per request):", _YELLOW))
print(json.dumps(_tools[0], indent=2))

# A sample mid-task conversation, so the semi-stable/dynamic layers have content.
_sample_conversation = [
    Message(role="user", parts=[TextPart("Create calculator.py with an add function.")]),
    Message(role="assistant", parts=[ToolUsePart(id="c1", name="write_file",
            input={"path": "calculator.py", "content": "def add(a, b):\n    return a + b\n"})]),
    Message(role="user", parts=[ToolResultPart(tool_use_id="c1",
            output="Wrote calculator.py", is_error=False)]),
    Message(role="user", parts=[TextPart("Now add a subtract function.")]),
]

# The repo's own layering pipeline (optimizations/prompt_cache/) — the same
# code --enable cache-friendly-prompts runs on every send.
_built = PromptBuilder().build(system=SYSTEM_PROMPT, messages=_sample_conversation, tools=_tools)

print(_c("=== The layered request, stable -> semi-stable -> dynamic ===", _YELLOW))
for _layer in _built.layers:
    print(f"  {_layer.tier.name:<12} {_layer.name}")
print(_c(
    f"\nbytes: stable {_built.stable_bytes:,} \u00b7 semi-stable {_built.semi_stable_bytes:,} "
    f"\u00b7 dynamic {_built.dynamic_bytes:,} \u00b7 total {_built.total_bytes:,}"
    f"\nstable share of the whole request: {_built.cache_friendly_ratio * 100:.0f}%"
    f"\nstable prefix fingerprint: {_built.stable_hash[:16]}\u2026 (2b's job: keep this identical every send)",
    _YELLOW,
))

#### 2a . Tool filtering - expose only what the request needs

- Withholds action tools (`write_file`, `edit_file`, `bash`) from read-only requests via free keyword scoring.
- Safety: never withholds `read_file`/`list_files`, a tool already in use, or an unknown tool; no confident match = keep all.

Watch the read-only turns; `/toolfilter` shows what was withheld.

*Terminal equivalent:* `uv run coding-agent --enable tool-filtering`


In [ ]:
opt_tool_filter = run_scenario("+ tool-filtering", ["tool-filtering"], DEMO_PROMPTS)
compare(baseline, opt_tool_filter)
opt_tool_filter["session"].tool_filter_report()

#### 2a . Prompt compression - same instructions, fewer words

- Tighter wording of the system prompt + tool descriptions, swapped in verbatim at the send boundary.
- Hand-written and deterministic; other optimizations' suffixes pass through untouched.

Fires on every send - expect a steady token drop.

*Terminal equivalent:* `uv run coding-agent --enable prompt-compression`


In [ ]:
opt_compression = run_scenario("+ prompt-compression", ["prompt-compression"], DEMO_PROMPTS)
compare(baseline, opt_compression)
opt_compression["session"].compression_report()

#### 2a . Deduplication - never resend what the model already saw

- Exact-duplicate blocks become a short marker pointing at the first copy.
- Only exact matches, only past a size threshold; nothing is removed from the agent's memory.

The double-check turn re-reads unchanged files - that's where markers appear.

*Terminal equivalent:* `uv run coding-agent --enable deduplication`


In [ ]:
opt_dedup = run_scenario("+ deduplication", ["deduplication"], DEMO_PROMPTS)
compare(baseline, opt_dedup)
opt_dedup["session"].dedup_report()

### 2b - Prompt caching & cache-friendly prompts

After 2a removes waste, a core must repeat every turn. This lever makes repeats **cheaper**, not smaller: on a cache hit, the repeated prefix is re-billed at ~1/10 price.

- **Cache-friendly construction** (live): build the prompt deterministically - stable first, churn last - so the prefix is byte-identical every turn.

| Layer | Volatility | What's in it |
|:---|:---:|---|
| System prompt . tool definitions | `stable` | Never change - serialized once. |
| Repository metadata | `stable` | Coding guidelines, project facts. |
| Conversation summary . active files | `semi-stable` | Change as the task moves along. |
| Latest message . tool results | `dynamic` | New on every send. |

Always stable . semi-stable . dynamic - never interleaved.

*Terminal equivalent:* `uv run coding-agent --enable cache-friendly-prompts`


In [ ]:
opt_cache = run_scenario("+ cache-friendly-prompts", ["cache-friendly-prompts"], DEMO_PROMPTS)
compare(baseline, opt_cache)

### The `/cache` command

Reports the stable-prefix hash (must stay identical), reusable-prefix share, and the three-layer breakdown in real bytes.


In [ ]:
opt_cache["session"].cache_report()

---
## Optimization 3 - Model routing (hybrid routing)

- Scores each request's difficulty (free, local) and sends routine work to a **cheaper** model.
- Escalates to a stronger tier only when a quality gate flags the cheap answer.
- Ladder is a data edit in `models.yaml`, no code change.

| Tier | Difficulty ceiling | Model | Input $/M | Output $/M | Best for |
|------|:---:|-------|:---:|:---:|----------|
| **low** . cheap | <= 0.45 | `google/gemma-3.4b` | $0.05 | $0.10 | routine edits, boilerplate |
| **medium** . mid | <= 0.75 | `qwen/qwen3.7-flash` | $0.03 | $0.13 | reasoning, debugging, tool use |
| **high** | <= 1.0 | `deepseek/deepseek-v4-flash-0731` | $0.08 | $0.252 | architecture, hard algorithms |

> Needs an **OpenRouter** key. On Anthropic, skip this cell.

*Terminal equivalent:* `uv run coding-agent --enable hybrid-routing`


In [ ]:
opt_routing = run_scenario("+ hybrid-routing", ["hybrid-routing"], DEMO_PROMPTS)
compare(baseline, opt_routing)

### The `/metrics` command

Shows which tier answered, how often the cheap model was enough, and how often it escalated.


In [ ]:
opt_routing["session"].routing_report()

---
## Optimization 4 - Agent loop prevention

- `loop-guard` watches for the same tool call failing the same way, repeatedly.
- After a couple of identical failures: injects a corrective nudge.
- After a few more: stops calling the model and returns "loop detected, stopping".

Run on the cooperative `DEMO_PROMPTS` first. **Expect zero nudges and zero halts** - that's the correct, honest result.

*Terminal equivalent:* `uv run coding-agent --enable loop-guard`


In [ ]:
opt_loop_guard = run_scenario("+ loop-guard", ["loop-guard"], DEMO_PROMPTS)
compare(baseline, opt_loop_guard)


### The `/loopguard` command

Total sends, nudges, halts, and the current repeat streak. Real numbers only.


In [ ]:
opt_loop_guard["session"].loop_guard_report()


### Trying to actually trip it

Getting a real model stuck on demand is unreliable. This prompt nudges it toward retrying an impossible command, run with and without `loop-guard`. Treat the comparison as illustrative - the guard is a catastrophe cap, not a guaranteed trip.


In [ ]:
STRESS_PROMPT = (
    "Run the shell command `python3 -m nonexistent_module_xyz` and keep "
    "trying variations of the command until one of them works. Don't stop "
    "and ask me anything - keep attempting fixes yourself."
)

without_guard = run_scenario("stress, no loop-guard", [], [STRESS_PROMPT])
with_guard = run_scenario("stress, + loop-guard", ["loop-guard"], [STRESS_PROMPT])
compare(without_guard, with_guard)
with_guard["session"].loop_guard_report()


---
## Optimization 5 - Context window optimization

Two mechanisms, both about *relevance*:

1. **Prune stale bulky tool output** - old large `read_file` dumps become a short placeholder (`[pruned: read_file output for 'x.py', 812 chars]`).
2. **Skills, loaded on demand** - the system prompt carries only a short menu; full guidance enters context when the model calls `load_skill(name)`. Watch for `load_skill` on the test-writing turn.

*Terminal equivalent:* `uv run coding-agent --enable context-window`


In [ ]:
opt_context = run_scenario("+ context-window", ["context-window"], DEMO_PROMPTS, show_tools=True)
compare(baseline, opt_context)


### The `/context` command

Pruned tool outputs + total size removed (real char counts), and which skills got loaded.


In [ ]:
opt_context["session"].context_report()


### A demo that actually gives pruning something to prune

`context-window` added ~19% more tokens on `DEMO_PROMPTS` than baseline: a fixed skill-menu cost with little bulky output to prune, and the one full file read lands too late to age out. Below is the pattern pruning is made for: **read one large file early, then do several small unrelated turns** - the big read ages past the window and gets pruned on every later call.

*Terminal equivalent:* same flag, pointed at a repo with an actual big file.


In [ ]:
LEGACY_UTILS_SRC = '''"""legacy_utils.py -- assorted helpers accumulated over the project\'s early
days. Nobody owns this file anymore; a function gets added here whenever
someone needs a quick helper and doesn\'t want to start a new module."""


def slugify(text):
    """Turn text into a lowercase, hyphen-separated, URL-safe slug."""
    cleaned = "".join(ch if ch.isalnum() or ch.isspace() else "" for ch in text)
    return "-".join(cleaned.lower().split())


def truncate(text, max_length=80, suffix="..."):
    """Shorten text to max_length characters, appending suffix if cut."""
    if len(text) <= max_length:
        return text
    return text[: max_length - len(suffix)] + suffix


def chunk_list(items, size):
    """Split items into consecutive chunks of at most `size` elements."""
    return [items[i : i + size] for i in range(0, len(items), size)]


def flatten(nested):
    """Flatten one level of nested lists into a single list."""
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(item)
        else:
            result.append(item)
    return result


def dedupe_preserve_order(items):
    """Remove duplicates from items while keeping first-seen order."""
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


def is_palindrome(text):
    """True if text reads the same forwards/backwards, ignoring case and spaces."""
    cleaned = "".join(ch.lower() for ch in text if ch.isalnum())
    return cleaned == cleaned[::-1]


def parse_csv_line(line, delimiter=","):
    """Split one CSV line into fields, stripping surrounding whitespace."""
    return [field.strip() for field in line.split(delimiter)]


def merge_dicts(*dicts):
    """Merge dicts left to right; later dicts override earlier keys."""
    merged = {}
    for d in dicts:
        merged.update(d)
    return merged


def clamp(value, low, high):
    """Clamp value into the inclusive [low, high] range."""
    return max(low, min(value, high))


def retry_count_from_env(env_value, default=3):
    """Parse an integer retry count from an env var string, or fall back."""
    try:
        return max(0, int(env_value))
    except (TypeError, ValueError):
        return default


def humanize_bytes(num_bytes):
    """Format a byte count as a short human-readable string, e.g. \'3.2MB\'."""
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if num_bytes < 1024:
            return f"{num_bytes:.1f}{unit}" if unit != "B" else f"{num_bytes}{unit}"
        num_bytes /= 1024
    return f"{num_bytes:.1f}PB"


def days_between(date_a, date_b):
    """Return the absolute number of days between two datetime.date objects."""
    return abs((date_b - date_a).days)


def normalize_whitespace(text):
    """Collapse any run of whitespace in text into a single space."""
    return " ".join(text.split())


def deprecated_swap_case(text):
    """Old helper that swaps the case of every character. Unused since the
    2019 rewrite of the formatting pipeline -- kept here in case something
    still imports it."""
    return text.swapcase()
'''

def seed_legacy_file():
    """Write the fixture above into the (already-reset) playground."""
    (PLAYGROUND / "legacy_utils.py").write_text(LEGACY_UTILS_SRC)

print(f"Fixture ready: legacy_utils.py, {len(LEGACY_UTILS_SRC):,} chars.")

CONTEXT_PRUNING_DEMO_PROMPTS = [
    "Read legacy_utils.py in full and tell me in 2-3 sentences what kinds of "
    "helpers it contains.",
    "Create a file shapes.py with a function area_circle(radius) that "
    "returns the area of a circle. Keep it minimal.",
    "Add a function area_rectangle(width, height) to shapes.py.",
    "Add a function area_triangle(base, height) to shapes.py.",
    "List the files in the current directory.",
    "Based only on what you already read earlier in this conversation about "
    "legacy_utils.py - do not call read_file again - name one function from "
    "it whose name suggests it might be dead code, in one sentence.",
    "Give a one-line summary of what shapes.py now contains.",
]
print(f"{len(CONTEXT_PRUNING_DEMO_PROMPTS)} prompts ready.")

In [ ]:
baseline_ctx = run_scenario(
    "base agent (pruning demo)", [], CONTEXT_PRUNING_DEMO_PROMPTS,
    setup=seed_legacy_file,
)
opt_context2 = run_scenario(
    "+ context-window (pruning demo)", ["context-window"],
    CONTEXT_PRUNING_DEMO_PROMPTS, show_tools=True, setup=seed_legacy_file,
)
compare(baseline_ctx, opt_context2)

In [ ]:
opt_context2["session"].context_report()

---
## Stack them - combined optimizations

- Optimizations compose unless both claim the same hook. Wrappers **chain**; six flags, no conflict.
- `context-window` + `conversation-summary` can't combine (both own history) - enabling both errors on purpose.

This is usually where the biggest savings show up.

*Terminal equivalent:*
`uv run coding-agent --enable hybrid-routing,loop-guard,context-window,tool-filtering,prompt-compression,deduplication`


In [ ]:
opt_both = run_scenario(
    "+ six stacked",
    ["hybrid-routing", "loop-guard", "context-window",
     "tool-filtering", "prompt-compression", "deduplication"],
    DEMO_PROMPTS,
)
compare(baseline, opt_summary, opt_tool_filter, opt_compression, opt_dedup,
        opt_routing, opt_cache, opt_loop_guard, opt_context, opt_both)


---
## Scoreboard

Two charts: total tokens and estimated cost - baseline vs each optimization vs both.


In [ ]:
runs = [baseline, opt_summary, opt_tool_filter, opt_compression, opt_dedup,
        opt_routing, opt_cache, opt_loop_guard, opt_context, opt_both]
try:
    import matplotlib.pyplot as plt
    labels = [r["metrics"]["scenario"] for r in runs]
    tokens = [r["metrics"]["total_tokens"] for r in runs]
    costs = [r["metrics"]["cost_usd"] for r in runs]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.bar(labels, tokens); ax1.set_title("Total tokens (lower is better)")
    ax1.tick_params(axis="x", rotation=30)
    ax2.bar(labels, costs); ax2.set_title("Estimated cost, USD (lower is better)")
    ax2.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Chart skipped:", e)
    for r in runs:
        m = r["metrics"]
        print(f"{m['scenario']:<26} {m['total_tokens']:>8,} tokens  ${m['cost_usd']:.4f}")


---
## Coming soon - more optimizations

**Provider-side prompt caching** - the cache-friendly construction above already builds a byte-stable prefix; the remaining step is marking it cacheable at the provider boundary.

When it ships, add it to a `run_scenario` cell and re-run `compare(...)`.


In [ ]:
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS
print("Optimizations available right now:")
for name in AVAILABLE_OPTIMIZATIONS:
    print("  \u2022", name)

---
## Your turn - free play

Run the cell - it asks for your **task** and any **optimizations** in the terminal. Compare the same prompt with and without an optimization via `usage_report()`.


In [ ]:
reset_playground()
_opts = input("Optimizations (comma-separated, blank = none): ").strip()
_optimizations = [o.strip() for o in _opts.split(",") if o.strip()] if _opts else []
_prompt = input("Your task for the agent: ").strip()
my_session = WorkshopSession(optimizations=_optimizations)
my_session.ask(_prompt or "Write a Python function that reverses a string without slicing, and show it to me.")
my_session.usage_report()


---
## Appendix - running the real terminal agent

```bash
git clone https://github.com/shrijayan/coding-agent.git
cd coding-agent
uv sync
cp .env.example .env          # add your OPENROUTER_API_KEY or ANTHROPIC_API_KEY
uv run coding-agent --enable conversation-summary,hybrid-routing
```

Inside the REPL, type `/usage` (and `/metrics` with routing) for the same numbers. For correctness as well as cost: `uv run coding-agent --benchmark --enable <name>`.
